# Vaizdų generavimo programų kūrimas (OpenAI)

LLM modeliai yra ne tik teksto generavimui. Taip pat galima generuoti vaizdus pagal teksto aprašymus. Vaizdai kaip modalumas yra naudingi MedTech, architektūros, turizmo, žaidimų kūrimo, rinkodaros srityse ir ne tik. Šiame pamokoje dirbsime su šiandieniniais **GPT Image** modeliais OpenAI platformoje.

## Mokymosi tikslai

Pamokos pabaigoje galėsite:

- Paaiškinti, kas yra vaizdų generavimas ir kur jis yra naudingas.
- Suprasti `gpt-image` modelių šeimą ir kuo ji skiriasi nuo senesnių DALL·E modelių.
- Sukurti vaizdų generavimo programą ir redaguoti vaizdus.

## Kas yra vaizdų generavimas?

Vaizdų generavimo modeliai sukuria vaizdus pagal teksto užklausą. Šiuolaikiniai modeliai, tokie kaip `gpt-image`, treniruotės metu mokosi teksto ir vaizdų ryšio, o tada žingsnis po žingsnio atsitiktinį triukšmą paverčia į vaizdą, atitinkantį jūsų užklausą.

Dvi gerai žinomos vaizdų modelių šeimos yra:

- **`gpt-image` (OpenAI)** — dabartinė kartos modelis, naudojamas šioje pamokoje. Jis palaiko teksto į vaizdą generavimą ir vaizdų redagavimą (uždažymą su kauke).
- **Midjourney** — populiarus trečiosios šalies modelis su savo paslauga ir Discord pagrindu veikiančiu darbo procesu.

> Senesni OpenAI vaizdų modeliai — **DALL·E 2** ir **DALL·E 3** — yra paveldėti. DALL·E 3 jau nebėra prieinamas naujai diegimams, o funkcijos kaip `create_variation` egzistavo tik DALL·E 2. Naudokite `gpt-image` modelius naujoms programoms.

> **Svarbu:** `gpt-image` modeliai gražina sugeneruotą vaizdą kaip **base64** (`b64_json`), ne kaip URL. Jūsų kodas iššifruoja base64 eilutę į baitus ir jį išsaugo — nėra vaizdo URL atsisiuntimui.


## Kaip sukurti pirmąją vaizdų generavimo programėlę

Tad ko reikia, kad sukurtumėte vaizdų generavimo programėlę? Jums reikės šių bibliotekų:

- **python-dotenv**, labai rekomenduojama naudoti šią biblioteką, kad slaptažodžius laikytumėte *.env* faile, atskirai nuo kodo.
- **openai**, šią biblioteką naudosite bendraudami su OpenAI API.
- **pillow**, darbui su paveikslėliais Python kalboje.


1. Sukurkite *.env* failą su tokiu turiniu:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Surinkite aukščiau nurodytas bibliotekas faile, pavadintame *requirements.txt*, tokiu formatu:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Toliau sukurkite virtualią aplinką ir įdiekite bibliotekas:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!PASTABA]
> „Windows“ sistemoje naudokite šias komandas, kad sukurtumėte ir aktyvuotumėte savo virtualią aplinką:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Į failą, pavadintą *app.py*, pridėkite šį kodą:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # importuoti dotenv
    dotenv.load_dotenv()

    # Sukurkite OpenAI objektą (nuskaito OPENAI_API_KEY iš jūsų .env)
    client = openai.OpenAI()


    try:
        # Sukurkite paveikslėlį naudodami vaizdo generavimo API
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Įveskite savo eilutės tekstą čia
            size='1024x1024',
            n=1
        )
        # Nustatykite katalogą saugomam paveikslėliui
        image_dir = os.path.join(os.curdir, 'images')

        # Jei katalogas neegzistuoja, sukurkite jį
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicializuokite paveikslėlio kelią (atkreipkite dėmesį, kad bylos tipas turėtų būti png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image modeliai grąžina paveikslėlį kaip base64 (b64_json), ne URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Rodyti paveikslėlį numatytajame paveikslėlių peržiūrėtoje
        image = Image.open(image_path)
        image.show()

    # pagauti išimtis
    except openai.BadRequestError as err:
        print(err)

    ```

Paaiškinkime šį kodą:

- Pirmiausia importuojame reikalingas bibliotekas, įskaitant OpenAI biblioteką, dotenv biblioteką, base64 modulį ir Pillow biblioteką.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Po to sukuriame klientą, kuris skaito API raktą iš jūsų ``.env`` failo.

    ```python
    # Sukurkite OpenAI objektą
    client = openai.OpenAI()
    ```

- Toliau generuojame paveikslėlį:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Įrašykite savo užklausos tekstą čia
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` modeliai grąžina paveikslėlį kaip **base64** eilutę `data[0].b64_json`. Mes ją dekoduojame į baitus ir įrašome į failą — nėra URL, iš kur būtų galima atsisiųsti.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Galiausiai atidarome paveikslėlį ir naudojame standartinį vaizdų peržiūros įrankį jį parodyti:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Daugiau informacijos apie paveikslėlio generavimą

Pažiūrėkime į `images.generate` parametrus:

- **model**, yra naudotinas paveikslėlio modelis (pvz., `gpt-image-1`).
- **prompt**, yra tekstinė užklausa paveikslėlio generavimui. Čia tai "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, yra sugeneruoto paveikslėlio dydis (pvz., `1024x1024`, `1536x1024`, `1024x1536` arba `"auto"`).
- **n**, yra sugeneruotų paveikslėlių skaičius. Čia sugeneruojamas vienas.

> Paveikslėlių modeliai neturi `temperature` parametro — tai yra teksto generavimo kontrolė. Norėdami įvairovės, pakartokite API iškvietimą; kad sumažintumėte įvairovę, padarykite savo užklausą tiksliau.

## Papildomos paveikslėlių generavimo galimybės

Jūs matėte, kaip keliais Python kodo eilutėmis sugeneruoti paveikslėlį. `gpt-image` modeliai taip pat gali **redaguoti** esamą paveikslėlį. Pateikdami paveikslėlį, pasirinktą **kaukę** (žyminti sritį, kurią norima keisti) ir užklausą, galite pakeisti paveikslėlio dalį — pavyzdžiui, pridėti burtininko skrybėlę mūsų zuikiui.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# redagavimai taip pat grąžinami kaip base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Pagrindinis paveikslėlis rodo tik triušį; galutiniame paveikslėlyje triušiui uždėta skrybėlė.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
